# Análise de Perplexidade de Prompts

Este notebook demonstra como calcular a perplexidade de prompts usando o modelo GPT-2.

A perplexidade mede a incerteza do modelo ao processar um texto. Valores menores indicam maior confiança do modelo.

## Objetivo

Aprender a avaliar a qualidade de prompts através da métrica de perplexidade, identificando quais prompts são mais claros e específicos para o modelo.


## Importação das Bibliotecas

- **torch**: Biblioteca para cálculos matemáticos, especialmente com modelos de aprendizado profundo
- **transformers**: Fornece ferramentas para trabalhar com modelos de linguagem pré-treinados, como o GPT-2
- **GPT2Tokenizer**: Converte texto em números que o modelo entende (tokenização)
- **GPT2LMHeadModel**: Modelo GPT-2 pré-treinado para processar texto


In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel


## Carregar Tokenizador e Modelo

O tokenizer transforma texto em sequência de números (tokens).

O modelo GPT-2 pré-treinado processa o texto e gera previsões.


In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Configurar padding token se necessário
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


## Função para Calcular Perplexidade

A função `calcular_perplexidade`:
- Recebe um texto (prompt) como entrada
- Converte o texto em tensores usando o tokenizer
- O modelo gera uma previsão e calcula a loss (medida de quão bem o modelo previu as próximas palavras)
- A perplexidade é obtida aplicando `exp()` à loss, transformando a perda em uma medida de incerteza

**Quanto menor a perplexidade, maior a confiança do modelo nas suas previsões.**


In [ ]:
def calcular_perplexidade(prompt):
    """
    Calcula a perplexidade de um prompt usando GPT-2.
    
    Args:
        prompt (str): Texto a ser analisado
        
    Returns:
        float: Valor da perplexidade (menor = mais confiante)
    """
    # Converter texto em tokens
    inputs = tokenizer(prompt, return_tensors='pt')
    
    # Calcular loss (quão bem o modelo previu as próximas palavras)
    outputs = model(**inputs, labels=inputs['input_ids'])
    loss = outputs.loss
    
    # Perplexidade = exp(loss)
    perplexidade = torch.exp(loss)
    
    return perplexidade.item()


## Comparar Prompts

Vamos comparar dois prompts:
- **Prompt 1**: Mais específico e detalhado
- **Prompt 2**: Mais genérico e curto

O prompt mais específico geralmente terá menor perplexidade, indicando que o modelo está mais confiante ao processá-lo.


In [ ]:
# Prompt 1: Mais específico e detalhado
prompt1 = "Descreva a estrutura e usos do ácido acético."

# Prompt 2: Mais genérico e curto
prompt2 = "Fale sobre o ácido acético."

# Calcular perplexidade de cada prompt
perplexidade1 = calcular_perplexidade(prompt1)
perplexidade2 = calcular_perplexidade(prompt2)


In [ ]:
print(f"Perplexidade do Prompt 1 (específico): {perplexidade1:.2f}")
print(f"Perplexidade do Prompt 2 (genérico): {perplexidade2:.2f}")
print(f"\nDiferença: {perplexidade2 - perplexidade1:.2f}")
print(f"Prompt 1 é {((perplexidade2 / perplexidade1) - 1) * 100:.1f}% mais confiável")


## Interpretação dos Resultados

- **Perplexidade mais baixa (Prompt 1)**: O modelo encontrou uma sequência de palavras que reconheceu mais facilmente. Portanto, está mais "confiante" ao processar este prompt.

- **Perplexidade mais alta (Prompt 2)**: O segundo prompt foi mais difícil para o modelo prever, o que pode ocorrer porque:
  - É mais curto e genérico
  - O modelo fica mais incerto sobre qual direção seguir
  - Falta contexto específico

**Conclusão**: Quanto mais informação e contexto fornecemos, maior a segurança do modelo em processar o prompt.
